## 0. Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from dataloader import prepare_data, VolatilityDataset, TICKERS
from RNNmodel import VanillaRNN
from train import run_training, evaluate

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1. Load Data

In [ ]:
returns, target_vol = prepare_data(force_download=False)

print('Returns shape:    ', returns.shape)    # (trading_days, num_stocks)
print('Target vol shape: ', target_vol.shape)
print('Date range:       ', returns.index[0], '→', returns.index[-1])

## 2. Chronological Train / Val / Test Splits


- **Train:** 2000–2017
- **Val:** 2018, starting at the 21st trading day (buffer for last training target window)
- **Test:** 2019–2020, starting at the 21st trading day of 2019

In [ ]:
# --- Slice DataFrames by date ---
train_returns = returns.loc[:'2017-12-31']
train_target  = target_vol.loc[:'2017-12-31']

# Val: all of 2018
val_returns = returns.loc['2018-01-01':'2018-12-31']
val_target  = target_vol.loc['2018-01-01':'2018-12-31']

# Test: 2019–2020 (same reasoning)
test_returns = returns.loc['2019-01-01':'2020-12-31']
test_target  = target_vol.loc['2019-01-01':'2020-12-31']

print(f'Train days: {len(train_returns)}  |  '
      f'Val days: {len(val_returns)}  |  '
      f'Test days: {len(test_returns)}')

## 3. Build Datasets


In [ ]:
LOOKBACK   = 60   # swap to 10 for the short-lookback experiment
BATCH_SIZE = 256

train_dataset = VolatilityDataset(train_returns, train_target, lookback=LOOKBACK)

train_dataset = VolatilityDataset(train_returns, train_target, lookback=LOOKBACK)
val_dataset   = VolatilityDataset(val_returns,   val_target,   lookback=LOOKBACK)
test_dataset  = VolatilityDataset(test_returns,  test_target,  lookback=LOOKBACK)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train samples: {len(train_dataset):,}')
print(f'Val samples:   {len(val_dataset):,}')
print(f'Test samples:  {len(test_dataset):,}')



## 4. Instantiate Model & Optimizer

In [ ]:
NUM_STOCKS  = len(TICKERS)   # 99
EMBED_DIM   = 16
HIDDEN_SIZE = 64
DROPOUT     = 0.1

model = VanillaRNN(
    num_stocks=NUM_STOCKS,
    embed_dim=EMBED_DIM,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTrainable parameters: {total_params:,}')

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)

## 5. Train

In [ ]:
history = run_training(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    device=DEVICE,
    num_epochs=30,
    patience=5,
    checkpoint_path=f'rnn_lookback{LOOKBACK}_best.pt',
    verbose=True,
)

## 6. Learning Curves

In [ ]:
epochs = range(1, len(history['train_mse']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, history['train_mse'], label='Train MSE')
axes[0].plot(epochs, history['val_mse'],   label='Val MSE')
axes[0].set_title('MSE (log-vol space)'); axes[0].legend()

axes[1].plot(epochs, history['val_mae'])
axes[1].set_title('Val MAE (original vol space)')

axes[2].plot(epochs, history['val_corr'])
axes[2].set_title('Val Pearson Correlation')

plt.tight_layout()
plt.savefig(f'learning_curves_lookback{LOOKBACK}.png', dpi=150)
plt.show()

## 7. Test Set Evaluation (full 2019–2020)

In [ ]:
test_metrics = evaluate(model, test_loader, DEVICE)

print('=== Test Results ===')
print(f'  MSE (log-vol):        {test_metrics["mse"]:.5f}')
print(f'  MAE (original vol):   {test_metrics["mae"]:.5f}')
print(f'  Pearson Correlation:  {test_metrics["correlation"]:.4f}')